# 11 — Federated learning: does collaboration help?

The forecasting question is closed (persistence unbeaten, 04–09), so the FL question here is **not** "beat persistence". It is the core federated question with the privacy motivation attached: *if each provider trains only on its own data, does averaging their models (FedAvg) recover what a centrally pooled model would learn — and does either beat purely local models?*

Setup, deliberately minimal and fully declared:

- **Clients** = the same C_110 stable public cohort as 04/07/08 (one client per provider). The full 392-provider `fl_client` pool across all specialties is the later scale-up.
- **Model** = linear on (current level, 1-month momentum) predicting next month's level — interpretable, and linear models make FedAvg exact to state: one round of sample-weighted coefficient averaging.
- **Splits and regimes unchanged**: train origins before Apr 2024 (full-history vs post-Covid from Mar 2020 — an external proposal to move the regime break to 2022 was rejected; regime boundaries are declared once), validate on the standard 1,407 FY 2024-25 rows (asserted identical to 04), sealed year untouched.
- **Persistence sits in the results table** — any FL claim is judged against it.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # lstsq fits and coefficient averaging
from pathlib import Path

PROC = Path("../data/processed")
TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - never read below
VAL_START = pd.Timestamp("2024-04-01")       # validation year FY 2024-25 (same as 04)
COVID = pd.Timestamp("2020-03-01")           # declared regime break (unchanged)

panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
dev = panel[panel["month"] < TEST_START]

df = dev[(dev["tfunc_code"] == "C_110") & dev["fl_client"] & dev["is_public"]].copy()
df = df.sort_values(["provider_code", "month"])
g = df.groupby("provider_code")["total_waiting"]
df["target"] = g.shift(-1)                   # next month's list
df["delta_1"] = df["total_waiting"] - g.shift(1)   # 1-month momentum
df = df.dropna(subset=["target"])

FEATS = ["total_waiting", "delta_1"]
train_full = df[df["month"] < VAL_START].dropna(subset=["delta_1"])   # each provider's first
train_post = train_full[train_full["month"] >= COVID]                 # row has no momentum yet
val = df[df["month"] >= VAL_START].copy()

assert val["month"].max() < TEST_START       # a sealed-year row leaking in would trip this
assert len(val) == 1407, len(val)            # must be the exact rows 04/07/08 scored
assert int(val["delta_1"].isna().sum()) == 1 # exactly one known gap trips otherwise: RJ8's first
                                             # C_110 return is May 2024 (96 months in other specialties)
val["delta_1"] = np.where(val["delta_1"].notna(), val["delta_1"], 0.0)
# with no prior C_110 month RJ8's momentum is unknown - zero change is the neutral reading
naive_mae = np.abs(val["target"] - val["total_waiting"]).mean()
assert round(naive_mae) == 153, naive_mae
print(len(train_full), "train-full,", len(train_post), "train-post,", len(val), "validation rows;",
      val["provider_code"].nunique(), "clients")

8895 train-full, 6184 train-post, 1407 validation rows; 128 clients


## Three ways to train the same model

**local** — each client fits on its own history only (no sharing at all). **fedavg** — clients share only their fitted coefficients; the server averages them weighted by client sample count (raw rows never leave the client). **pooled** — the privacy-free upper reference: one fit on all rows centrally. All three score the same validation rows.

In [2]:
# Goal: fit local / fedavg / pooled per regime, score everything on the same 1,407 rows.
def fit(rows):                               # least squares with an intercept column of 1s
    X = np.column_stack([np.ones(len(rows)), rows[FEATS].to_numpy()])
    beta, *_ = np.linalg.lstsq(X, rows["target"].to_numpy(), rcond=None)
    return beta

def predict(beta, rows):
    return np.column_stack([np.ones(len(rows)), rows[FEATS].to_numpy()]) @ beta

truth = np.concatenate([r["target"].to_numpy() for _, r in val.groupby("provider_code")])
current = np.concatenate([r["total_waiting"].to_numpy() for _, r in val.groupby("provider_code")])
# truth/current rebuilt through the same groupby order as the per-client predictions below,
# so every model's error vector lines up row for row

results = []
def score(pred, regime, model):
    err = truth - pred
    results.append({"regime": regime, "model": model, "mae": np.abs(err).mean(),
                    "rmse": np.sqrt((err ** 2).mean()), "n": len(err)})

for regime, tr in [("full", train_full), ("post", train_post)]:
    betas = {p: (fit(r), len(r)) for p, r in tr.groupby("provider_code")}
    assert set(val["provider_code"]) - set(betas) == {"RJ8"}
    # RJ8 is the one client with zero training rows (C_110 series starts May 2024): the FL
    # selling point in miniature - the shared model can serve it, local-only cannot and
    # falls back to persistence for its rows

    local = np.concatenate([predict(betas[p][0], r) if p in betas else r["total_waiting"].to_numpy()
                            for p, r in val.groupby("provider_code")])
    n_tot = sum(n for _, n in betas.values())
    beta_fed = sum(b * n for b, n in betas.values()) / n_tot   # FedAvg: weighted coefficient average
    fed = np.concatenate([predict(beta_fed, r) for _, r in val.groupby("provider_code")])
    pooled = np.concatenate([predict(fit(tr), r) for _, r in val.groupby("provider_code")])

    score(local, regime, "local-only")
    score(fed, regime, "fedavg")
    score(pooled, regime, "pooled")
score(current, "-", "naive persistence")

res = pd.DataFrame(results)
for _, r in res.sort_values(["regime", "mae"]).iterrows():
    print(f"{r['regime']:5s} {r['model']:18s} MAE {r['mae']:6,.1f}   RMSE {r['rmse']:6,.1f}   (n={r['n']})")

-     naive persistence  MAE  153.5   RMSE  330.1   (n=1407)
full  pooled             MAE  153.8   RMSE  330.3   (n=1407)
full  local-only         MAE  161.9   RMSE  345.2   (n=1407)
full  fedavg             MAE  186.5   RMSE  363.5   (n=1407)
post  pooled             MAE  155.4   RMSE  331.9   (n=1407)
post  local-only         MAE  166.5   RMSE  350.5   (n=1407)
post  fedavg             MAE  228.8   RMSE  401.4   (n=1407)


## Phase 2 — average deltas, not levels

The failure above is scale: intercepts fitted on raw levels (~50 to ~18,000) cannot be averaged. Two remedies, each one line of algebra: **(a) fedavg-delta** — every client fits *change* = a + b·momentum and the coefficients are averaged (changes are far closer in scale than levels); **(b) fedavg-delta + local intercept** — clients share only the slope b and keep their own intercept a: the shared part is scale-free, the scale stays local (the simplest personalization). RJ8 (no training rows) gets intercept 0 in (b) — its forecast leans on persistence, which is exactly what a cold-start client should do.

In [3]:
# Goal: does moving the federation into delta space fix the measured non-IID failure?
def fit_delta(rows):                         # change = a + b * last month's change
    X = np.column_stack([np.ones(len(rows)), rows["delta_1"].to_numpy()])
    beta, *_ = np.linalg.lstsq(X, (rows["target"] - rows["total_waiting"]).to_numpy(), rcond=None)
    return beta

for regime, tr in [("full", train_full), ("post", train_post)]:
    bd = {p: (fit_delta(r), len(r)) for p, r in tr.groupby("provider_code")}
    n_tot = sum(n for _, n in bd.values())
    bfed = sum(b * n for b, n in bd.values()) / n_tot

    fed_delta = np.concatenate([r["total_waiting"].to_numpy() + bfed[0] + bfed[1] * r["delta_1"].to_numpy()
                                for _, r in val.groupby("provider_code")])
    fed_local_a = np.concatenate([r["total_waiting"].to_numpy()
                                  + (bd[p][0][0] if p in bd else 0.0)   # RJ8: no local intercept yet
                                  + bfed[1] * r["delta_1"].to_numpy()
                                  for p, r in val.groupby("provider_code")])
    score(fed_delta, regime, "fedavg-delta")
    score(fed_local_a, regime, "fedavg-delta+local-a")

res = pd.DataFrame(results)
for _, r in res.sort_values(["regime", "mae"]).iterrows():
    print(f"{r['regime']:5s} {r['model']:20s} MAE {r['mae']:6,.1f}   RMSE {r['rmse']:6,.1f}   (n={r['n']})")

-     naive persistence    MAE  153.5   RMSE  330.1   (n=1407)
full  pooled               MAE  153.8   RMSE  330.3   (n=1407)
full  fedavg-delta         MAE  154.0   RMSE  332.0   (n=1407)
full  fedavg-delta+local-a MAE  155.3   RMSE  334.3   (n=1407)
full  local-only           MAE  161.9   RMSE  345.2   (n=1407)
full  fedavg               MAE  186.5   RMSE  363.5   (n=1407)
post  pooled               MAE  155.4   RMSE  331.9   (n=1407)
post  fedavg-delta         MAE  155.5   RMSE  333.1   (n=1407)
post  fedavg-delta+local-a MAE  157.4   RMSE  336.3   (n=1407)
post  local-only           MAE  166.5   RMSE  350.5   (n=1407)
post  fedavg               MAE  228.8   RMSE  401.4   (n=1407)


## Where collaboration helps: per-client view

The pooled numbers hide the distribution: FedAvg's promise is protecting *small* clients from overfitting their own short noisy history. Per client, does the shared model beat the local one?

In [4]:
# Goal: per-client MAE, local vs fedavg (full-history regime) - who gains from sharing?
tr = train_full
betas = {p: (fit(r), len(r)) for p, r in tr.groupby("provider_code")}
n_tot = sum(n for _, n in betas.values())
beta_fed = sum(b * n for b, n in betas.values()) / n_tot

rows = []
for p, r in val.groupby("provider_code"):
    t = r["target"].to_numpy()
    local_pred = predict(betas[p][0], r) if p in betas else r["total_waiting"].to_numpy()
    rows.append({"provider_code": p, "n_val": len(r),           # RJ8 (no training rows) falls
                 "n_train": betas[p][1] if p in betas else 0,   # back to persistence locally
                 "mae_local": np.abs(t - local_pred).mean(),
                 "mae_fedavg": np.abs(t - predict(beta_fed, r)).mean(),
                 "mae_naive": np.abs(t - r["total_waiting"].to_numpy()).mean()})
per = pd.DataFrame(rows)
print(f"fedavg beats local at {(per['mae_fedavg'] < per['mae_local']).mean():.1%} "
      f"of {len(per)} clients; local beats naive at {(per['mae_local'] < per['mae_naive']).mean():.1%}")
print("\nmedian per-client MAE: local %.0f | fedavg %.0f | naive %.0f"
      % (per["mae_local"].median(), per["mae_fedavg"].median(), per["mae_naive"].median()))

res.to_csv(PROC / "fl_summary_c110.csv", index=False)
per.to_csv(PROC / "fl_per_client_c110.csv", index=False)
print("\n", res.shape, "->", PROC / "fl_summary_c110.csv", "|", per.shape, "->", PROC / "fl_per_client_c110.csv")

fedavg beats local at 41.4% of 128 clients; local beats naive at 46.1%

median per-client MAE: local 113 | fedavg 134 | naive 113

 (11, 5) -> ..\data\processed\fl_summary_c110.csv | (128, 6) -> ..\data\processed\fl_per_client_c110.csv
